<a href="https://colab.research.google.com/github/LeeSeongJinn/Doosan_Rokey_bootcamp/blob/main/2_Applied_AI/Day14/2_Transformer_SST2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Transformer (DistilBERT) 감정분석 — GLUE/SST-2

In [ ]:
!pip uninstall -y torch torchvision torchaudio

# 2. 캐시를 무시하고 CUDA 12.4 버전으로 '강제 재설치' 합니다.
!pip install --no-cache-dir --force-reinstall torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

# 3. 나머지 Hugging Face 패키지들도 마저 설치합니다.
!pip -q install -U "datasets>=3.0.1" "transformers>=4.45.2" "accelerate>=1.0.1" "evaluate>=0.4.2"

Found existing installation: torch 2.11.0+cu128
Uninstalling torch-2.11.0+cu128:
  Successfully uninstalled torch-2.11.0+cu128
Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128
Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128
Looking in indexes: https://download.pytorch.org/whl/cu124
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 174.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 103.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 109.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 281.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 137.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 144.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 139.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.9/203.9 kB 23.2 MB/s eta 0:00:00
ERROR: Operation cancelled by user
^C


런타임 재시작

In [ ]:
import torch, transformers, datasets

print("PyTorch:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

PyTorch: 2.6.0+cu124
CUDA Available: True
Using device: cuda


In [ ]:
from datasets import load_dataset
ds = load_dataset("nyu-mll/glue", "sst2", trust_remote_code=True)
print(ds)

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'nyu-mll/glue' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'nyu-mll/glue' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


README.md:   0%|          | 0.00/35.3k [00:00<?, ?B/s]

sst2/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.11MB            

sst2/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

sst2/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 72.8kB            

...

sst2/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 872
    })
    test: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1821
    })
})


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding
import torch

# 1. 환경 설정
MODEL = "distilbert-base-uncased"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. 토크나이저 로드
tokenizer = AutoTokenizer.from_pretrained(MODEL)

# 3. 전처리 함수 (return_tensors는 여기서 하지 않고 collator에게 맡깁니다)
def preprocess(ex):
    return tokenizer(ex["sentence"], truncation=True, max_length=256)

# 4. 데이터셋 매핑 (ds가 정의되어 있다고 가정)
# remove_columns에 "label"은 포함하지 않도록 주의하세요!
enc = ds.map(preprocess, batched=True, remove_columns=["sentence", "idx"])

# 5. 데이터 콜레이터 (다이내믹 패딩 적용)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# 6. 모델 로드 (이때 발생하는 Warning은 무시해도 됩니다)
model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2).to(device)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/1821 [00:00<?, ? examples/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
# !pip install evaluate

  Using cached evaluate-0.4.6-py3-none-any.whl.metadata (9.5 kB)
Using cached evaluate-0.4.6-py3-none-any.whl (84 kB)


In [ ]:
import evaluate
from transformers import TrainingArguments, Trainer
import torch

# 지표 로드
acc = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def metrics(p):
    predictions, labels = p
    # 가장 높은 확률의 위치
    # .item(): 파이썬 숫자로 변경 (scalar array)
    # predictions의 shape 는 (데이터 개수, 클래스 개수)
    # 예) (데이터 개수, 클래스 개수)  (3,4)
    # axis = -1 마지막 축, 클래스 확률 들 중에서 가장 큰 값이 있는 위치(인덱스) 가져와
    # pred >> (3,)
    preds = predictions.argmax(-1)

    return {
        "acc": acc.compute(predictions=preds, references=labels)["accuracy"],
        "f1": f1.compute(predictions=preds, references=labels, average="binary")["f1"]
    }

# TrainingArguments 설정
args = TrainingArguments(
    output_dir="/content/sst2_2025",
    eval_strategy="epoch",          # 수정됨: evaluation_strategy -> eval_strategy
    save_strategy="epoch",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    learning_rate=2e-5,
    load_best_model_at_end=True,
    fp16=torch.cuda.is_available(), # GPU가 있을 때만 fp16 사용
    report_to="none",
    seed=2025
)
# fp16: 16bit / (default) 32bit float >> 16bit로 낮춤(메모리 절약_2배)
# 참고: A100 이상 돌리고 싶다면? bf16=True 사용 권고 (bf: brain floating point)

# Trainer 초기화
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=enc["train"],
    eval_dataset=enc["validation"],
    data_collator=data_collator,
    compute_metrics=metrics
)

In [ ]:
txt=["This movie was amazing!","Worst film ever."]
inp=tokenizer(txt,return_tensors="pt",padding=True,truncation=True,max_length=256).to(model.device)
with torch.no_grad(): out=torch.softmax(model(**inp).logits,dim=-1).cpu().numpy()
for t,p in zip(txt,out): print(f"{t}\n→ Negative={p[0]:.3f}, Positive={p[1]:.3f}")

This movie was amazing!
→ Negative=0.479, Positive=0.521
Worst film ever.
→ Negative=0.482, Positive=0.518
